# Day 09. Exercise 02
# Metrics

## 0. Imports

In [131]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import cross_val_score, ParameterGrid
import itertools
import joblib

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [132]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
day = pd.read_csv('../data/dayofweek.csv')
df = pd.concat([day['dayofweek'], df], axis=1)
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=21
)
df.head()

,dayofweek,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,4,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,4,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,4,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [133]:
def score(model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)
    res = round(pd.DataFrame({
        'accuracy': [accuracy_score(y_test, y_pred)],
        'percision': [precision_score(y_test, y_pred, average='weighted')],
        'recall': [recall_score(y_test, y_pred, average='weighted')],
        'roc_auc': [roc_auc_score(y_test, y_score, multi_class='ovo', average='weighted')]
        }), 5)
    print(res)
    return y_pred

In [134]:
y_pred = score(SVC(kernel='rbf', C=10, gamma='auto', class_weight=None, probability=True, random_state=21))

   accuracy  percision   recall  roc_auc
0   0.88757    0.89267  0.88757  0.97878


## 3. Decision tree

In [135]:
y_pred = score(DecisionTreeClassifier(class_weight='balanced', criterion='gini', max_depth=21, random_state=21).fit(X_train, y_train))

   accuracy  percision   recall  roc_auc
0   0.88462    0.88765  0.88462  0.93528


## 4. Random forest

In [136]:
y_pred = score(RandomForestClassifier(n_estimators=100, max_depth=24, criterion='entropy', class_weight='balanced', random_state=21).fit(X_train, y_train))

   accuracy  percision   recall  roc_auc
0   0.92604    0.92754  0.92604  0.98939


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [137]:
joblib.dump(RandomForestClassifier(n_estimators=100, max_depth=24, criterion='entropy', class_weight='balanced', random_state=21).fit(X_train, y_train),
            filename='RandomForestClassifier.joblib')
cm = confusion_matrix(y_test, y_pred)
res = []
for day in range(cm.shape[0]):
    total = cm[day].sum()
    error = total - cm[day,day]
    percent = round(error/total*100, 2)
    res.append({
        'total': total,
        'correct': cm[day,day],
        'error': error,
        'percent': percent})
pd.DataFrame(res)

,total,correct,error,percent
0,27,21,6,22.22
1,55,51,4,7.27
2,30,28,2,6.67
3,80,77,3,3.75
4,21,18,3,14.29
5,54,49,5,9.26
6,71,69,2,2.82


In [138]:
df_test = X_test.copy()
df_test['true_weekday'] = y_test.values
df_test['pred_weekday'] = y_pred
df_test['is_error'] = df_test['true_weekday'] != df_test['pred_weekday']

In [139]:
def calculate(desired):
    desired_col = [col for col in df_test.columns if col.startswith(f'{desired}_')]
    res = []
    for col in desired_col:
        true_col = df_test[df_test[col] == 1]
        if len(true_col) > 0:
            total = len(true_col)
            error = true_col['is_error'].sum()
            percent = round(error/total*100, 2)
            res.append({
            'labname': col.replace('labname_', ''),
            'total_samples': total,
            'errors': error,
            'error_percent': percent
        })
    return pd.DataFrame(res)

In [140]:
calculate('labname')

,labname,total_samples,errors,error_percent
0,code_rvw,13,1,7.69
1,lab03,1,1,100.00
2,lab03s,1,0,0.00
3,lab05s,6,1,16.67
4,laba04,35,6,17.14
5,laba04s,25,2,8.00
6,laba05,47,1,2.13
7,laba06,9,1,11.11
8,laba06s,15,2,13.33
9,project1,186,10,5.38


In [141]:
calculate('uid')

,labname,total_samples,errors,error_percent
0,uid_user_1,9,0,0.00
1,uid_user_10,12,1,8.33
2,uid_user_12,12,0,0.00
3,uid_user_13,17,1,5.88
4,uid_user_14,31,1,3.23
5,uid_user_15,2,0,0.00
6,uid_user_16,5,1,20.00
7,uid_user_17,7,0,0.00
8,uid_user_18,6,1,16.67
9,uid_user_19,19,2,10.53


## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [142]:
def score(model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)
    res = {
        'accuracy': [accuracy_score(y_test, y_pred)],
        'percision': [precision_score(y_test, y_pred, average='weighted')],
        'recall': [recall_score(y_test, y_pred, average='weighted')],
        'roc_auc': [roc_auc_score(y_test, y_score, multi_class='ovo', average='weighted')]
        }
    return res

In [143]:
score(RandomForestClassifier(n_estimators=100, max_depth=24, criterion='entropy', class_weight='balanced', random_state=21).fit(X_train, y_train))

{'accuracy': [0.9260355029585798],
 'percision': [0.9275374670957044],
 'recall': [0.9260355029585798],
 'roc_auc': [0.9893851880258296]}